# 🎵 MoodBeats AI — Session 4: FastAPI & Web Basics

So far we've built logic in Python. Today we connect it to a **web browser**.

---
## How the Web Works

```
Browser                  Your Python Code
  │  GET /               │
  │ ──────────────────►  │  @app.get('/')
  │                      │  def home(): ...
  │  HTML response        │
  │ ◄──────────────────  │  return HTML
  │                      │
[renders page]
```

1. Browser sends a **request** (GET or POST) to a URL
2. Your **FastAPI** server receives it, runs Python code
3. Server sends back **HTML** — the browser renders it

---
## Setup

In [ ]:
!pip install fastapi uvicorn jinja2 nest-asyncio -q
print("Ready!")

In [ ]:
# nest-asyncio lets FastAPI run inside Colab (normally Colab has its own event loop)
import nest_asyncio
nest_asyncio.apply()
print("nest-asyncio applied")

---
## Part 1: Your First FastAPI App

A **route** is a URL pattern mapped to a Python function.  
The `@app.get('/')` decorator tells FastAPI: "Call this function when someone requests GET /".

In [ ]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

app = FastAPI()

@app.get("/", response_class=HTMLResponse)
def home():
    return """
    <html>
    <body style="font-family: sans-serif; padding: 40px;">
        <h1>🎵 Welcome to MoodBeats AI!</h1>
        <p>Detect your mood, get music recommendations.</p>
        <a href="/detect">Try it now →</a>
    </body>
    </html>
    """

@app.get("/detect", response_class=HTMLResponse)
def detect_page():
    return """
    <html>
    <body style="font-family: sans-serif; padding: 40px;">
        <h1>Detect Your Mood</h1>
        <p>This page will have the mood input form.</p>
        <a href="/">← Back to home</a>
    </body>
    </html>
    """

print("App defined! Run the next cell to start the server.")

In [ ]:
# This starts the server — you'll see output. Stop it with the ⬛ button.
# In Colab, you can't click the link directly, but we'll test via requests below.
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server running on port 8000!")

In [ ]:
# Test the server
import requests

response = requests.get("http://localhost:8000/")
print(f"Status: {response.status_code}")
print(response.text[:200])

---
## Part 2: Jinja2 Templates — Separating HTML from Python

Putting HTML inside Python strings gets messy fast. **Jinja2 templates** are `.html` files where you can embed Python values.

```html
<!-- template.html -->
<h1>Hello, {{ name }}!</h1>
<p>You feel {{ emotion }}.</p>
```

In Python:
```python
return templates.TemplateResponse("template.html", {"request": request, "name": "Alice", "emotion": "happy"})
```

Jinja2 replaces `{{ name }}` with `"Alice"` at render time.

In [ ]:
import os

# Create a templates directory
os.makedirs("templates", exist_ok=True)

# Write a base template
with open("templates/base.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html>
<head>
    <title>MoodBeats AI</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-950 text-white min-h-screen">
    <!-- Navbar -->
    <nav class="bg-gray-900 px-6 py-4 flex items-center gap-6">
        <a href="/" class="text-indigo-400 font-bold text-lg">🎵 MoodBeats AI</a>
        <a href="/detect" class="text-gray-300 hover:text-white">Detect</a>
        <a href="/history" class="text-gray-300 hover:text-white">History</a>
    </nav>

    <!-- Page content injected here -->
    <main class="max-w-4xl mx-auto px-6 py-10">
        {% block content %}{% endblock %}
    </main>
</body>
</html>
""")

# Write landing page
with open("templates/landing.html", "w") as f:
    f.write("""
{% extends "base.html" %}
{% block content %}
<div class="text-center py-20">
    <h1 class="text-5xl font-bold text-indigo-400 mb-4">MoodBeats AI</h1>
    <p class="text-xl text-gray-400 mb-2">Welcome, {{ username }}!</p>
    <p class="text-gray-500 mb-8">Detect your mood. Discover your music.</p>
    <a href="/detect"
       class="bg-indigo-600 hover:bg-indigo-500 text-white px-8 py-3 rounded-lg font-semibold">
        Detect My Mood →
    </a>
</div>
{% endblock %}
""")

print("Templates written!")

In [ ]:
from fastapi import FastAPI, Request
from fastapi.templating import Jinja2Templates
import uvicorn, threading

app2 = FastAPI()
templates = Jinja2Templates(directory="templates")

@app2.get("/")
def home(request: Request):
    return templates.TemplateResponse("landing.html", {
        "request":  request,
        "username": "Student",   # We'll get this from sessions later
    })

thread2 = threading.Thread(target=lambda: uvicorn.run(app2, host="0.0.0.0", port=8001, log_level="error"), daemon=True)
thread2.start()

resp = requests.get("http://localhost:8001/")
print(f"Status: {resp.status_code}")
print("Page contains 'MoodBeats AI':", "MoodBeats AI" in resp.text)

### ✏️ In-Class Exercise 4a — Add a Detect Template

In [ ]:
# TODO: Create templates/detect.html that extends base.html
# It should contain:
#   - A heading: 'Detect Your Mood'
#   - A paragraph showing {{ error }} if it exists (use {% if error %})
#   - A simple text area with a button (the form action will be added in Session 5)

with open("templates/detect.html", "w") as f:
    f.write("""
{% extends "base.html" %}
{% block content %}
<h1 class="text-3xl font-bold mb-6">Detect Your Mood</h1>

{% if error %}
<div class="bg-red-900 text-red-200 p-4 rounded-lg mb-6">{{ error }}</div>
{% endif %}

<!-- TODO: Add a textarea and submit button here -->

{% endblock %}
""")

print("detect.html created!")

# TODO: Add a GET /detect route to app2 that renders detect.html
@app2.get("/detect")
def detect_page(request: Request, error: str = None):
    return templates.TemplateResponse("detect.html", {"request": request, "error": error})

resp = requests.get("http://localhost:8001/detect")
print(f"Detect page status: {resp.status_code}")

### ✏️ In-Class Exercise 4b — Pass a Variable from Route to Template

In [ ]:
# TODO: Modify the GET / route to pass a 'features' list to landing.html
# features = ["Webcam detection", "Text mood analysis", "8 music recommendations", "History tracking"]
# In landing.html, add a {% for feature in features %} loop to display them

# Hint: update templates/landing.html first, then update the route

---
## 🏠 After-Class Exercises

---
### After-Class A: Add an About Page

In [ ]:
# TODO: Create templates/about.html that extends base.html
# Add a GET /about route that renders it
# The page should describe what MoodBeats AI does

### After-Class B: Template Loop

Pass a list of emotions from the route and display them on the landing page.

In [ ]:
# In the GET / route, add:
# "emotions": ["😊 Happy", "😢 Sad", "😤 Angry", "😲 Surprised", "😐 Neutral"]
#
# In landing.html, add a {% for e in emotions %} loop to show an emotion strip
# (This is exactly what the real MoodBeats landing page does!)

### After-Class C (Challenge): 404 Error Page

In [ ]:
from fastapi import Request
from fastapi.responses import HTMLResponse
from fastapi.exceptions import HTTPException

# TODO: Register a 404 handler using @app2.exception_handler(404)
# It should render a friendly error page
@app2.exception_handler(404)
async def not_found(request: Request, exc: HTTPException):
    # TODO: return a TemplateResponse for error.html, or just return an HTMLResponse
    pass

---
## ✅ Session 4 Complete!

**You learned:**
- How the client-server model works
- FastAPI routes with `@app.get()`
- Jinja2 templates with variables, loops, and inheritance
- Tailwind CSS utility classes

**Next session:** Forms and POST requests — the detect page comes to life!